# Stage 2 · L1pre — preliminary all-cell clustering

Integrate all cells using the coarse tissue anchors (thresholded on `dist_geosk_pc`) to get a preliminary all-cell atlas and rough major-type labels.

These are the original pipeline notebooks, concatenated in execution order with paths normalized to `ENTEX_ROOT`. They document the method and run per tissue / major type across the full raw dataset (heavy compute); they are shown for reference and are **not re-executed in the book**. Example group shown where templated.

In [1]:
# === Reproduction setup ===
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT); import repro_guard


## 2a · mCG all-cell integration

_Source: `clustering/merged/L1pre/01.5kCG_clustering.ipynb`_

In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
from glob import glob

import anndata
import scanpy as sc
import scanpy.external as sce
from sklearn.preprocessing import normalize

from ALLCools.clustering import *
from ALLCools.plot import *
from ALLCools.integration.seurat_class import SeuratIntegration

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'


In [3]:
def dump_embedding(adata, name, n_dim=2):
    # put manifold coordinates into adata.obs
    for i in range(n_dim):
        adata.obs[f'{name}_{i}'] = adata.obsm[f'X_{name}'][:, i]
    return adata


In [4]:
indir = f'{ENTEX_ROOT}/clustering/merged/'
outdir = f'{ENTEX_ROOT}/clustering/'


In [5]:
# mcad = anndata.read_h5ad(f'{indir}cell_86689_16tissue_5kCG_autosomal.h5ad')
# mcad = anndata.AnnData(obs=mcad.obs, obsm=mcad.obsm)
# mcad.write_h5ad('cell_86689_16tissue_5kCG_autosomal.h5ad')


In [6]:
mcad = anndata.read_h5ad('cell_86689_16tissue_5kCG_autosomal.h5ad')
mcad


In [7]:
mcad.obsm['lsi_all'] = mcad.obsm['5kCG_pca'].copy()
npc = significant_pc_test(mcad, p_cutoff=0.01, obsm='lsi_all', update=False)


In [8]:
npc = 50
mcad.obsm['X_lsi'] = normalize(mcad.obsm['lsi_all'][:, :npc], axis=1)


In [9]:
tsne(mcad, obsm='X_lsi', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
mcad.obsm[f'5kCG_u{npc}_tsne'] = mcad.obsm['X_tsne'].copy()


In [10]:
ds = 0.2
coord_base = 'tsne'
npc = 50
mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'5kCG_u{npc}_{coord_base}'].copy()
dump_embedding(mcad, coord_base)

fig, axes = plt.subplots(2, 3, figsize=(15, 8), dpi=300, constrained_layout=True)

tmp = mcad.obs.copy()
ax = axes[0, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # show_legend=True
                       )

ax = axes[0, 1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Tissue',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        show_legend=True
                       )

ax = axes[0, 2]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1}, 
                        # show_legend=True
                       )


ax = axes[1, 0]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue=np.log10(tmp['FinalmCReads']+1),
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )

ax = axes[1, 1]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue='mCGFrac',
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )

ax = axes[1, 2]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue='mCHFrac',
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )


In [11]:
ds = 0.2
coord_base = 'tsne'
npc = 80
mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'5kCG_u{npc}_{coord_base}'].copy()
dump_embedding(mcad, coord_base)

fig, axes = plt.subplots(2, 3, figsize=(15, 8), dpi=300, constrained_layout=True)

tmp = mcad.obs.copy()
ax = axes[0, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # show_legend=True
                       )

ax = axes[0, 1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Tissue',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        show_legend=True
                       )

ax = axes[0, 2]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1}, 
                        # show_legend=True
                       )


ax = axes[1, 0]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue=np.log10(tmp['FinalmCReads']+1),
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )

ax = axes[1, 1]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue='mCGFrac',
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )

ax = axes[1, 2]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue='mCHFrac',
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )


In [12]:
sample_list = np.sort(mcad.obs['Donor'].astype(str).unique())
sample_list

In [13]:
adata_list = [mcad[mcad.obs['Donor']==xx] for xx in sample_list]


In [14]:
anchor_index = {}
for xx,adata in zip(sample_list, adata_list):
    tmp = pd.DataFrame(index=adata.obs.index).reset_index().reset_index()
    anchor_index[xx] = tmp.set_index('cell')['index'].to_dict()


In [15]:
indir = f'{ENTEX_ROOT}/'
tissue_list = [xx.split('/')[-2] for xx in glob(f'{indir}tissue/*/')]
tissue_list


In [16]:
anchor = []
for t in tissue_list:
    anchor_file = glob(f'{outdir}tissue/{t}/mCG_5kb_seurat_anchor_*.hdf')[0]
    adata_file = glob(f'{outdir}tissue/{t}/mCG_5kb_lsi.h5ad')[0]
    anchor_tmp = pd.read_hdf(anchor_file)
    adata_tmp = anndata.read_h5ad(adata_file)
    sample_tmp = adata_tmp.obs['Donor'].unique()
    cell_list = [adata_tmp.obs.index[adata_tmp.obs['Donor']==xx] for xx in sample_tmp]
    thres = np.percentile(anchor_tmp['dist_geosk_pc'], 60)
    anchor_tmp = anchor_tmp.loc[anchor_tmp['dist_geosk_pc']<thres, ['x1','x2','score']].copy()
    anchor_tmp['x1'] = cell_list[0][anchor_tmp['x1']].map(anchor_index[sample_tmp[0]])
    anchor_tmp['x2'] = cell_list[1][anchor_tmp['x2']].map(anchor_index[sample_tmp[1]])
    anchor_tmp['x1_donor'] = sample_tmp[0]
    anchor_tmp['x2_donor'] = sample_tmp[1]
    anchor.append(anchor_tmp)
    
anchor = pd.concat(anchor, axis=0)



In [17]:
anchor[['x1_donor','x2_donor']].value_counts().sort_index()


In [18]:
n = len(adata_list)
anchor_df = {}

for i in range(n - 1):
    for j in range(i + 1, n):
        x1, x2 = sample_list[[i,j]]
        selc = (anchor['x1_donor']==x1) & (anchor['x2_donor']==x2)
        anchor_df[(i,j)] = anchor.loc[selc, ['x1', 'x2', 'score']]
        if selc.sum()==0:
            selc = (anchor['x1_donor']==x2) & (anchor['x2_donor']==x1)
            anchor_df[(i,j)] = anchor.rename({'x1':'x2','x2':'x1'}, axis=1).loc[selc, ['x1', 'x2', 'score']]
        

In [19]:
integrator = SeuratIntegration()


In [20]:
integrator.adata_dict = {k: v for k, v in zip(list(range(len(adata_list))), adata_list)}
integrator.n_dataset = len(adata_list)
integrator.n_cells = [adata.shape[0] for adata in adata_list]

# intra-dataset KNN for scoring the anchors
integrator.k_local = None
integrator.key_local = 'X_lsi'
integrator._calculate_local_knn()

# alignments and all_pairs
integrator.alignments = None
integrator._get_all_pairs()
integrator.anchor = anchor_df


In [21]:
npc = 50
corrected = integrator.integrate(key_correct='lsi_all',
                                 row_normalize=True,
                                 n_components=npc,
                                 k_weight=100,
                                 sd=1)


In [22]:
corrected = pd.DataFrame(normalize(np.concatenate(corrected, axis=0), axis=1), 
                         index=np.concatenate([xx.obs.index for xx in adata_list]))

mcad.obsm[f'5kCG_u{npc}_seuratfilter60'] = corrected.loc[mcad.obs.index].values
mcad.obsm[f'5kCG_u{npc}_seuratfilter60'] = normalize(mcad.obsm[f'5kCG_u{npc}_seuratfilter60'][:, :npc], axis=1)

tsne(mcad, obsm=f'5kCG_u{npc}_seuratfilter60', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
mcad.obsm[f'5kCG_u{npc}_seuratfilter60_tsne'] = mcad.obsm['X_tsne'].copy()


In [23]:
ds = 0.2
coord_base = 'tsne'
mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'5kCG_u{npc}_seuratfilter60_{coord_base}'].copy()
dump_embedding(mcad, coord_base)

fig, axes = plt.subplots(2, 3, figsize=(15, 8), dpi=300, constrained_layout=True)

tmp = mcad.obs.copy()
ax = axes[0, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # show_legend=True
                       )

ax = axes[0, 1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Tissue',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        show_legend=True
                       )

ax = axes[0, 2]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1}, 
                        # show_legend=True
                       )


ax = axes[1, 0]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue=np.log10(tmp['FinalmCReads']+1),
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )

ax = axes[1, 1]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue='mCGFrac',
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )

ax = axes[1, 2]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue='mCHFrac',
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )


In [24]:
mcad

In [25]:
ds = 0.2
coord_base = 'tsne'
mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'5kCG_u{npc}hm_{coord_base}'].copy()
dump_embedding(mcad, coord_base)

fig, axes = plt.subplots(2, 3, figsize=(15, 8), dpi=300, constrained_layout=True)

tmp = mcad.obs.copy()
ax = axes[0, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # show_legend=True
                       )

ax = axes[0, 1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Tissue',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        show_legend=True
                       )

ax = axes[0, 2]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1}, 
                        # show_legend=True
                       )


ax = axes[1, 0]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue=np.log10(tmp['FinalmCReads']+1),
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )

ax = axes[1, 1]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue='mCGFrac',
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )

ax = axes[1, 2]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue='mCHFrac',
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )


In [26]:
mcad.write_h5ad('5kCG_lsi.h5ad')


## 2b · 3C all-cell integration

_Source: `clustering/merged/L1pre/02.100k3C_clustering.ipynb`_

In [27]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
from glob import glob

import anndata
import scanpy as sc
import scanpy.external as sce
from sklearn.preprocessing import normalize

from ALLCools.clustering import *
from ALLCools.plot import *
from ALLCools.integration.seurat_class import SeuratIntegration

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'


In [28]:
def dump_embedding(adata, name, n_dim=2):
    # put manifold coordinates into adata.obs
    for i in range(n_dim):
        adata.obs[f'{name}_{i}'] = adata.obsm[f'X_{name}'][:, i]
    return adata


In [29]:
indir = f'{ENTEX_ROOT}/clustering/merged/'
outdir = f'{ENTEX_ROOT}/clustering/'


In [30]:
mcad = anndata.read_h5ad(f'{indir}cell_86689_16tissue_100k3C_autosomal.h5ad')
# mcad = anndata.AnnData(obs=mcad.obs, obsm=mcad.obsm)
# mcad.write_h5ad('cell_86689_16tissue_100k3C_autosomal.h5ad')
mcad

In [31]:
# mcad = anndata.read_h5ad('cell_86689_16tissue_5kCG_autosomal.h5ad')
# mcad


In [32]:
npc = significant_pc_test(mcad, p_cutoff=0.05, obsm='100k3C_pca', update=False)


In [33]:
from sklearn.decomposition import TruncatedSVD

model = TruncatedSVD(n_components=50, algorithm='arpack')
# matrix = np.concatenate(matrix, axis=1)
mcad.obsm['100k3C_pca'] = model.fit_transform(mcad.X)


In [34]:
npc = 50
mcad.obsm['X_pca'] = normalize(mcad.obsm['100k3C_pca'][:, :npc], axis=1)


In [35]:
tsne(mcad, obsm='X_pca', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
mcad.obsm[f'100k3C_pc{npc}_tsne'] = mcad.obsm['X_tsne'].copy()


In [36]:
ds = 0.2
coord_base = 'tsne'
mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'100k3C_pc{npc}_{coord_base}'].copy()
dump_embedding(mcad, coord_base)

fig, axes = plt.subplots(2, 3, figsize=(15, 8), dpi=300, constrained_layout=True)

tmp = mcad.obs.copy()
ax = axes[0, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # show_legend=True
                       )

ax = axes[0, 1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Tissue',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        show_legend=True
                       )

ax = axes[0, 2]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1}, 
                        # show_legend=True
                       )


ax = axes[1, 0]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue=np.log10(tmp['FinalmCReads']+1),
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )

ax = axes[1, 1]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue='mCGFrac',
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )

ax = axes[1, 2]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue='mCHFrac',
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )


In [37]:
ds = 0.2
coord_base = 'tsne'
mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'100k3C_u{npc}_{coord_base}'].copy()
dump_embedding(mcad, coord_base)

fig, axes = plt.subplots(2, 3, figsize=(15, 8), dpi=300, constrained_layout=True)

tmp = mcad.obs.copy()
ax = axes[0, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # show_legend=True
                       )

ax = axes[0, 1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Tissue',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        show_legend=True
                       )

ax = axes[0, 2]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1}, 
                        # show_legend=True
                       )


ax = axes[1, 0]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue=np.log10(tmp['FinalmCReads']+1),
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )

ax = axes[1, 1]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue='mCGFrac',
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )

ax = axes[1, 2]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue='mCHFrac',
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )


In [38]:
sample_list = np.sort(mcad.obs['Donor'].astype(str).unique())
sample_list

In [39]:
adata_list = [mcad[mcad.obs['Donor']==xx] for xx in sample_list]


In [40]:
anchor_index = {}
for xx,adata in zip(sample_list, adata_list):
    tmp = pd.DataFrame(index=adata.obs.index).reset_index().reset_index()
    anchor_index[xx] = tmp.set_index('cell_id')['index'].to_dict()


In [41]:
indir = f'{ENTEX_ROOT}/'
tissue_list = np.sort([xx.split('/')[-2] for xx in glob(f'{indir}tissue/*/')])
print(len(tissue_list))


In [42]:
anchor = []
for t in tissue_list:
    anchor_file = glob(f'{outdir}tissue/{t}/HiC_100kb_seurat_anchor_*.hdf')[0]
    adata_file = glob(f'{outdir}tissue/{t}/HiC_100kb_pca.h5ad')[0]
    anchor_tmp = pd.read_hdf(anchor_file)
    adata_tmp = anndata.read_h5ad(adata_file)
    sample_tmp = np.sort(adata_tmp.obs['Donor'].unique())
    cell_list = [adata_tmp.obs.index[adata_tmp.obs['Donor']==xx] for xx in sample_tmp]
    thres = np.percentile(anchor_tmp['dist_geosk_pc'], 60)
    anchor_tmp = anchor_tmp.loc[anchor_tmp['dist_geosk_pc']<thres, ['x1','x2','score']].copy()
    anchor_tmp['x1'] = cell_list[0][anchor_tmp['x1']].map(anchor_index[sample_tmp[0]])
    anchor_tmp['x2'] = cell_list[1][anchor_tmp['x2']].map(anchor_index[sample_tmp[1]])
    anchor_tmp['x1_donor'] = sample_tmp[0]
    anchor_tmp['x2_donor'] = sample_tmp[1]
    anchor.append(anchor_tmp)
    
anchor = pd.concat(anchor, axis=0)



In [43]:
anchor[['x1_donor','x2_donor']].value_counts().sort_index()


In [44]:
n = len(adata_list)
anchor_df = {}

for i in range(n - 1):
    for j in range(i + 1, n):
        x1, x2 = sample_list[[i,j]]
        selc = (anchor['x1_donor']==x1) & (anchor['x2_donor']==x2)
        anchor_df[(i,j)] = anchor.loc[selc, ['x1', 'x2', 'score']]
        if selc.sum()==0:
            selc = (anchor['x1_donor']==x2) & (anchor['x2_donor']==x1)
            anchor_df[(i,j)] = anchor.rename({'x1':'x2','x2':'x1'}, axis=1).loc[selc, ['x1', 'x2', 'score']]
        

In [45]:
integrator = SeuratIntegration()


In [46]:
integrator.adata_dict = {k: v for k, v in zip(list(range(len(adata_list))), adata_list)}
integrator.n_dataset = len(adata_list)
integrator.n_cells = [adata.shape[0] for adata in adata_list]

# intra-dataset KNN for scoring the anchors
integrator.k_local = None
integrator.key_local = 'X_pca'
integrator._calculate_local_knn()

# alignments and all_pairs
integrator.alignments = None
integrator._get_all_pairs()
integrator.anchor = anchor_df


In [47]:
npc = 50
corrected = integrator.integrate(key_correct='100k3C_pca',
                                 row_normalize=True,
                                 n_components=npc,
                                 k_weight=100,
                                 sd=1)


In [48]:
# corrected = pd.DataFrame(normalize(np.concatenate(corrected, axis=0), axis=1), 
#                          index=np.concatenate([xx.obs.index for xx in adata_list]))

# mcad.obsm[f'100k3C_u{npc}_seuratfilter60'] = corrected.loc[mcad.obs.index].values
# mcad.obsm[f'100k3C_u{npc}_seuratfilter60'] = normalize(mcad.obsm[f'100k3C_u{npc}_seuratfilter60'][:, :npc], axis=1)

# tsne(mcad, obsm=f'100k3C_u{npc}_seuratfilter60', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
# mcad.obsm[f'100k3C_u{npc}_seuratfilter60_tsne'] = mcad.obsm['X_tsne'].copy()


In [49]:
# ds = 0.2
# coord_base = 'tsne'
# mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'100k3C_u{npc}_seuratfilter60_{coord_base}'].copy()
# dump_embedding(mcad, coord_base)

# fig, axes = plt.subplots(2, 3, figsize=(15, 8), dpi=300, constrained_layout=True)

# tmp = mcad.obs.copy()
# ax = axes[0, 0]
# _ = categorical_scatter(data=tmp,
#                         ax=ax,
#                         coord_base=coord_base,
#                         hue='Donor',
#                         s=ds,
#                         labelsize=8,
#                         max_points=None,
#                         palette='tab20',
#                         scatter_kws={'rasterized':True},
#                         # show_legend=True
#                        )

# ax = axes[0, 1]
# _ = categorical_scatter(data=tmp,
#                         ax=ax,
#                         coord_base=coord_base,
#                         hue='Tissue',
#                         s=ds,
#                         labelsize=8,
#                         max_points=None,
#                         palette='tab20',
#                         scatter_kws={'rasterized':True},
#                         show_legend=True
#                        )

# ax = axes[0, 2]
# _ = categorical_scatter(data=tmp,
#                         ax=ax,
#                         coord_base=coord_base,
#                         hue='celltype',
#                         text_anno='celltype', 
#                         s=ds,
#                         labelsize=8,
#                         max_points=None,
#                         palette='tab20',
#                         scatter_kws={'rasterized':True},
#                         # legend_kws={'ncol':1}, 
#                         # show_legend=True
#                        )


# ax = axes[1, 0]
# _ = continuous_scatter(ax=ax,
#                        data=tmp,
#                        hue=np.log10(tmp['FinalmCReads']+1),
#                        coord_base=coord_base,
#                        max_points=None,
#                        labelsize=8,
#                        s=ds
#                   )

# ax = axes[1, 1]
# _ = continuous_scatter(ax=ax,
#                        data=tmp,
#                        hue='mCGFrac',
#                        coord_base=coord_base,
#                        max_points=None,
#                        labelsize=8,
#                        s=ds
#                   )

# ax = axes[1, 2]
# _ = continuous_scatter(ax=ax,
#                        data=tmp,
#                        hue='mCHFrac',
#                        coord_base=coord_base,
#                        max_points=None,
#                        labelsize=8,
#                        s=ds
#                   )


In [50]:
corrected = pd.DataFrame(normalize(np.concatenate(corrected, axis=0), axis=1), 
                         index=np.concatenate([xx.obs.index for xx in adata_list]))

mcad.obsm[f'100k3C_pc{npc}_seuratfilter60'] = corrected.loc[mcad.obs.index].values
mcad.obsm[f'100k3C_pc{npc}_seuratfilter60'] = normalize(mcad.obsm[f'100k3C_pc{npc}_seuratfilter60'][:, :npc], axis=1)

tsne(mcad, obsm=f'100k3C_pc{npc}_seuratfilter60', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
mcad.obsm[f'100k3C_pc{npc}_seuratfilter60_tsne'] = mcad.obsm['X_tsne'].copy()


In [51]:
ds = 0.2
coord_base = 'tsne'
mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'100k3C_pc{npc}_seuratfilter60_{coord_base}'].copy()
dump_embedding(mcad, coord_base)

fig, axes = plt.subplots(2, 3, figsize=(15, 8), dpi=300, constrained_layout=True)

tmp = mcad.obs.copy()
ax = axes[0, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # show_legend=True
                       )

ax = axes[0, 1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Tissue',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        show_legend=True
                       )

ax = axes[0, 2]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1}, 
                        # show_legend=True
                       )


ax = axes[1, 0]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue=np.log10(tmp['FinalmCReads']+1),
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )

ax = axes[1, 1]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue='mCGFrac',
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )

ax = axes[1, 2]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue='mCHFrac',
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )


In [52]:
mcad.write_h5ad('100k3C_pca.h5ad')
mcad

## 2c · joint embedding → preliminary L1

_Source: `clustering/merged/L1pre/03.joint_embed.ipynb`_

In [53]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

import anndata
import scanpy as sc
import scanpy.external as sce
from sklearn.preprocessing import normalize
from sklearn.decomposition import TruncatedSVD

from ALLCools.clustering import *
from ALLCools.plot import *
from ALLCools.integration.seurat_class import SeuratIntegration

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'


In [54]:
def dump_embedding(adata, name, n_dim=2):
    # put manifold coordinates into adata.obs
    for i in range(n_dim):
        adata.obs[f'{name}_{i}'] = adata.obsm[f'X_{name}'][:, i]
    return adata


In [55]:
group_name = 'All'

In [56]:
adata_mc = anndata.read_h5ad('5kCG_lsi.h5ad')
adata_3c = anndata.read_h5ad('100k3C_pca.h5ad')
adata_3c = adata_3c[adata_mc.obs.index].copy()


In [57]:
npc_cg = [int(xx.split('_')[1][1:]) for xx in adata_mc.obsm.keys() if '_seuratfilter60_tsne' in xx][0]
# npc_3c = [int(xx.split('_')[1][1:]) for xx in adata_3c.obsm.keys() if '100k3C_u' in xx][0]
npc_3c = [int(xx.split('_')[1][2:]) for xx in adata_3c.obsm.keys() if '_seuratfilter60_tsne' in xx][0]
print(npc_cg, npc_3c)


In [58]:
adata = anndata.AnnData(obs=adata_mc.obs)
# adata.obsm[f'5kCG100k3C_u{npc_cg}u{npc_3c}'] = np.hstack([normalize(adata_mc.obsm[f'5kCG_u{npc_cg}hm'], axis=1),
#                                                           normalize(adata_3c.obsm['X_pca'], axis=1)])
adata.obsm[f'5kCG100k3C_u{npc_cg}pc{npc_3c}'] = np.hstack([normalize(adata_mc.obsm[f'5kCG_u{npc_cg}_seuratfilter60'], axis=1),
                                                          normalize(adata_3c.obsm[f'100k3C_pc{npc_3c}_seuratfilter60'], axis=1)])
tsne(adata, obsm=f'5kCG100k3C_u{npc_cg}pc{npc_3c}', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
adata.obsm[f'5kCG100k3C_u{npc_cg}pc{npc_3c}_tsne'] = adata.obsm['X_tsne'].copy()


In [59]:
ds = 0.2
coord_base = 'tsne'
adata.obsm[f'X_{coord_base}'] = adata.obsm[f'5kCG100k3C_u{npc_cg}pc{npc_3c}_{coord_base}'].copy()
dump_embedding(adata, coord_base)

fig, axes = plt.subplots(2, 3, figsize=(15, 8), dpi=300, constrained_layout=True)

tmp = adata.obs.copy()
ax = axes[0, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # show_legend=True
                       )

ax = axes[0, 1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Tissue',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        show_legend=True
                       )

ax = axes[0, 2]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1}, 
                        # show_legend=True
                       )


ax = axes[1, 0]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue=np.log10(tmp['FinalmCReads']+1),
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )

ax = axes[1, 1]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue='mCGFrac',
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )

ax = axes[1, 2]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue='mCHFrac',
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )

plt.savefig(f'{group_name}_5kCG100k3C_meta.pdf', transparent=True)


In [60]:
adata.write_h5ad('5kCG100k3C_embed.h5ad')


In [61]:
adata = anndata.read_h5ad('5kCG100k3C_embed.h5ad')


In [62]:
# clustering name
clustering_name = 'L1'

# ConsensusClustering
# Important factores
n_neighbors = 25
leiden_resolution = 1.0
# this parameter is the final target that limit the total number of clusters
# Higher accuracy means more conservative clustering results and less number of clusters
target_accuracy = 0.95
min_cluster_size = 100

# Other ConsensusClustering parameters
metric = 'euclidean'
consensus_rate = 0.8
leiden_repeats = 500
random_state = 0
train_frac = 0.5
train_max_n = 500
max_iter = 50
n_jobs = 32

# Dendrogram via Multiscale Bootstrap Resampling
nboot = 10000
method_dist = 'correlation'
method_hclust = 'average'

plot_type = 'static'

In [63]:
cc = ConsensusClustering(model=None,
                         n_neighbors=n_neighbors,
                         metric=metric,
                         min_cluster_size=min_cluster_size,
                         leiden_repeats=leiden_repeats,
                         leiden_resolution=leiden_resolution,
                         consensus_rate=consensus_rate,
                         random_state=random_state,
                         train_frac=train_frac,
                         train_max_n=train_max_n,
                         max_iter=max_iter,
                         n_jobs=n_jobs,
                         target_accuracy=target_accuracy)


In [64]:
# cc.add_data(adata.obsm[f'5kCG100k3C_u{npc_cg}u{npc_3c}'])
# cc.compute_neighbors()

In [65]:
# import warnings
# from collections import OrderedDict
# from concurrent.futures import ProcessPoolExecutor, as_completed

# import anndata
# import joblib
# import leidenalg
# import matplotlib.pyplot as plt
# import networkx as nx
# import numpy as np
# import pandas as pd
# import seaborn as sns
# from imblearn.ensemble import BalancedRandomForestClassifier
# from natsort import natsorted
# from scanpy.neighbors import Neighbors
# from sklearn.metrics import (
#     adjusted_rand_score,
#     balanced_accuracy_score,
#     confusion_matrix,
#     pairwise_distances,
# )
# from sklearn.model_selection import cross_val_predict
# from ALLCools.clustering.ConsensusClustering import _leiden_runner

# partition_type=None
# partition_kwargs=None
# use_weights=True
# n_iterations=-1

# g = cc._neighbors.to_igraph()

# # generate n different seeds for each single leiden partition
# np.random.seed(cc.random_state)
# leiden_repeats = cc.leiden_repeats
# n_jobs = cc.n_jobs
# random_states = np.random.choice(range(99999), size=leiden_repeats, replace=False)
# step = max(int(leiden_repeats / n_jobs), 10)
# random_state_chunks = [random_states[i : min(i + step, leiden_repeats)] for i in range(0, leiden_repeats, step)]

# results = []
# print(f"Repeating leiden clustering {leiden_repeats} times")
# with ProcessPoolExecutor(max_workers=n_jobs) as executor:
#     future_dict = {}
#     for random_state_chunk in random_state_chunks:
#         # flip to the default partition type if not over writen by the user
#         if partition_type is None:
#             partition_type = leidenalg.RBConfigurationVertexPartition
#         # prepare find_partition arguments as a dictionary, appending to whatever the user provided
#         # it needs to be this way as this allows for the accounting of a None resolution
#         # (in the case of a partition variant that doesn't take it on input)
#         if partition_kwargs is None:
#             partition_kwargs = {}
#         else:
#             if "seed" in partition_kwargs:
#                 print("Warning: seed in the partition_kwargs will be ignored, use seed instead.")
#                 del partition_kwargs["seed"]
#         if use_weights:
#             partition_kwargs["weights"] = np.array(g.es["weight"]).astype(np.float64)
#         partition_kwargs["n_iterations"] = n_iterations
#         partition_kwargs["resolution_parameter"] = cc.leiden_resolution
#         # clustering proper
#         future = executor.submit(
#             _leiden_runner,
#             g=g,
#             random_states=random_state_chunk,
#             partition_type=partition_type,
#             **partition_kwargs,
#         )
#         future_dict[future] = random_state_chunks

#     for future in as_completed(future_dict):
#         _ = future_dict[future]
#         try:
#             data = future.result()
#             results.append(data)
#         except Exception as exc:
#             print(f"_leiden_runner generated an exception: {exc}")
#             raise exc
# total_result = pd.concat(results, axis=1, sort=True)
# cc.leiden_result_df = total_result
# cluster_count = cc.leiden_result_df.apply(lambda i: i.unique().size)
# print(
#     f"Found {cluster_count.min()} - {cluster_count.max()} clusters, "
#     f"mean {cluster_count.mean():.1f}, std {cluster_count.std():.2f}"
# )


In [66]:
# cc.save('ConcensusClustering.model.lib')


In [67]:
# import joblib
# cc = joblib.load('ConcensusClustering.model.lib')


In [68]:
# # data: row is leiden run, column is cell
# data = cc.leiden_result_df.T
# data

In [69]:
# # group cell into raw clusters if their hamming distance < 1 - consensus_rate
# import time
# cur_cluster_id = 0
# clusters = {}
# start_time = time.time()
# while data.shape[1] > 1:
#     # print('pop', time.time() - start_time)
#     seed_cell = data.pop(data.columns[0])
#     # print('copy', time.time() - start_time)
#     # tmp1 = 
#     # tmp2 = 
#     # print('dist', time.time() - start_time)
#     distance = (data.values!=seed_cell.values[:,None]).sum(axis=0) / data.shape[0]
#     # print('judge', time.time() - start_time)
#     judge = distance < (1 - cc.consensus_rate)
#     # print('list', time.time() - start_time)
#     this_cluster_cells = [seed_cell.name] + data.columns[judge].to_list()
#     # print('assign', time.time() - start_time)
#     for cell in this_cluster_cells:
#         clusters[cell] = cur_cluster_id
#     # print('remove', time.time() - start_time)
#     data = data.loc[:, ~judge].copy()
#     cur_cluster_id += 1
#     print(cur_cluster_id, data.shape[1], time.time() - start_time)
#     if time.time() - start_time > 300:
#         break
    

In [70]:
# if data.shape[1] == 1:
#     # if there is only one cell remain
#     clusters[data.columns[0]] = cur_cluster_id
# clusters = pd.Series(clusters).sort_index()

# # renumber clusters based on cluster size
# counts = clusters.value_counts()
# cluster_map = {c: i for i, c in enumerate(counts.index)}
# clusters = clusters.map(cluster_map)
# # renumber small clusters as -1
# counts = clusters.value_counts()
# small_clusters = counts[counts < cc.min_cluster_size].index
# clusters[clusters.isin(small_clusters)] = -1

# print(f"{(clusters != -1).sum()} cells assigned to {clusters.unique().size - 1} raw clusters")
# print(f"{(clusters == -1).sum()} cells are multi-leiden outliers")
# cc._multi_leiden_clusters = clusters.values


In [71]:
cc.fit_predict(adata.obsm[f'5kCG100k3C_u{npc_cg}pc{npc_3c}'])

In [72]:
adata.obs['leiden_cons'] = cc.label.copy()
adata.obs['leiden_cv'] = cc.cv_predicted_label.copy()


In [73]:
cc.save('ConcensusClustering.model.lib')
adata.write_h5ad('5kCG100k3C_embed.h5ad')


In [74]:
from sklearn.metrics import adjusted_rand_score as ARI
ARI(adata.obs['leiden_cons'], adata.obs['leiden_cv'])


In [75]:
adata = anndata.read_h5ad('5kCG100k3C_embed.h5ad')


In [76]:
ds = 0.2
coord_base = 'tsne'
adata.obsm[f'X_{coord_base}'] = adata.obsm[f'5kCG100k3C_u{npc_cg}pc{npc_3c}_{coord_base}'].copy()
dump_embedding(adata, coord_base)

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

tmp = adata.obs.copy()
ax = axes[0,0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='leiden_cons',
                        text_anno='leiden_cons',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # show_legend=True
                       )

ax = axes[0,1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='leiden_cv',
                        text_anno='leiden_cv',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # show_legend=True
                       )

ax = axes[1,0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Tissue',
                        # text_anno='celltype', 
                        s=ds,
                        labelsize=6,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True
                       )

ax = axes[1,1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype', 
                        s=ds,
                        labelsize=6,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1}, 
                        # show_legend=True
                       )

# plt.savefig(f'{group_name}_5kCG100k3C_cluster.pdf', transparent=True)


In [77]:
# from glob import glob
# indir = f'{ENTEX_ROOT}/'
# tissue_list = np.sort([xx.split('/')[-2] for xx in glob(f'{indir}tissue/*/')])
# tissue_list

In [78]:
# ds = 0.2
# coord_base = 'tsne'
# ncol = 2
# nrow = 8
# fig, axes = plt.subplots(nrow, ncol, figsize=(5*ncol,4*nrow), dpi=300, constrained_layout=True)
# for i,t in enumerate(tissue_list):
#     ax = axes.flatten()[i]
#     ax.scatter(adata.obs['tsne_0'], adata.obs['tsne_1'], s=ds, edgecolor='none', 
#                rasterized=True, color=(0.8,0.8,0.8))
#     tmp = adata.obs[adata.obs['Tissue']==t]
#     _ = categorical_scatter(data=tmp,
#                             ax=ax,
#                             s=ds*5,
#                             coord_base='tsne',
#                             hue='ClusterTissue',
#                             text_anno='ClusterTissue',
#                             labelsize=6,
#                             palette='tab20',
#                             #show_legend=True
#                            )
#     ax.set_title(t, fontsize=15, fontweight='bold')

# plt.savefig(f'{group_name}_5kCG100k3C_tissuecluster.pdf', transparent=True)


In [79]:
adata_mc.obsm[f'X_{coord_base}'] = adata_mc.obsm[f'5kCG_u{npc_cg}_seuratfilter60_{coord_base}'].copy()
dump_embedding(adata_mc, coord_base)
adata_3c.obsm[f'X_{coord_base}'] = adata_3c.obsm[f'100k3C_pc{npc_3c}_seuratfilter60_{coord_base}'].copy()
dump_embedding(adata_3c, coord_base)


In [80]:
## seurat coordinate

fig, axes = plt.subplots(2, 3, figsize=(15, 8), dpi=300, constrained_layout=True)

for i,xx in enumerate([adata, adata_mc, adata_3c]):
    dump_embedding(xx, coord_base)
    tmp = xx.obs.copy()
    tmp['leiden_cons'] = adata.obs['leiden_cons'].copy()
    ax = axes[0, i]
    _ = categorical_scatter(data=tmp,
                            ax=ax,
                            coord_base=coord_base,
                            hue='Tissue',
                            s=ds,
                            labelsize=8,
                            max_points=None,
                            palette='tab20',
                            scatter_kws={'rasterized':True},
                            show_legend=True
                           )

    ax = axes[1, i]
    _ = categorical_scatter(data=tmp,
                            ax=ax,
                            coord_base=coord_base,
                            hue='leiden_cons',
                            text_anno='leiden_cons', 
                            s=ds,
                            labelsize=8,
                            max_points=None,
                            palette='tab20',
                            scatter_kws={'rasterized':True},
                            # legend_kws={'ncol':1}, 
                            # show_legend=True
                           )

for xx,ax in zip(['mCG+3C', 'mCG', '3C'], axes[0]):
    ax.set_title(xx, fontsize=15)
    

In [81]:
# ## harmony coordinate

# fig, axes = plt.subplots(2, 3, figsize=(15, 8), dpi=300, constrained_layout=True)

# for i,xx in enumerate([adata, adata_mc, adata_3c]):
#     dump_embedding(xx, coord_base)
#     tmp = xx.obs.copy()
#     tmp['leiden_cons'] = adata.obs['leiden_cons'].copy()
#     ax = axes[0, i]
#     _ = categorical_scatter(data=tmp,
#                             ax=ax,
#                             coord_base=coord_base,
#                             hue='Donor',
#                             s=ds,
#                             labelsize=8,
#                             max_points=None,
#                             palette='tab20',
#                             scatter_kws={'rasterized':True},
#                             # show_legend=True
#                            )

#     ax = axes[1, i]
#     _ = categorical_scatter(data=tmp,
#                             ax=ax,
#                             coord_base=coord_base,
#                             hue='leiden_cons',
#                             text_anno='leiden_cons', 
#                             s=ds,
#                             labelsize=8,
#                             max_points=None,
#                             palette='tab20',
#                             scatter_kws={'rasterized':True},
#                             # legend_kws={'ncol':1}, 
#                             # show_legend=True
#                            )


In [82]:
leg = ['Hema Myeloid', 'Hema Mast', 
       'Hema B', 'Hema Tnaive', 'Hema Tmem', 'Hema NK', 
       'Glia Astro', 'Glia Oligo', 
       'Neu Exc', 'Neu Inh', 'Neu Schw', 
       'Epi Endcri', 'Epi Duc', 'Epi Aci', 'Epi Krt/Lum', 'Epi Alv', 'Epi TPB', 
       'Epi Gas', 'Epi AdrCtx', 'Epi Ent', 'Epi BrstBasal', 
       'Mus Skl', 'Mus Crd', 
       'Endo Lym', 'Endo Ves', 'SmMus/Peri', 
       'Fibro HT', 'Fibro GI', 'Fibro B', 'Fibro EndN', 'Fibro EpiN', 
       'Fibro Sk', 'Fibro Mus', 'Fibro Adr', 'Fibro Myo']



In [83]:
# tmp = anndata.read_h5ad(f'{ENTEX_ROOT}/clustering/merged/L1/5kCG100k3C_embed.h5ad')
# tmp = tmp.obs[['L1']]
# tmp['leiden_cons'] = adata.obs['leiden_cons'].copy()
# count = tmp.value_counts().unstack().fillna(0)
# count = count.loc[leg]


In [84]:
# ratio = count / count.sum(axis=0)
# idxcol = np.argsort(np.argmax(ratio.values, axis=0))
# fig, ax = plt.subplots(figsize=(12,6))
# sns.heatmap(ratio.iloc[:, idxcol], xticklabels=1, yticklabels=1, ax=ax)

In [85]:
selc = [[1, 62, 59, 53, 45, 38, 24], [26], 
        [39, 15, 31], [11], [29, 25, 0], [30], 
        [41], [58, 14], 
        [12, 43, 54], [32, 37], [18], 
        [8], [40], [19], [48, 42, 46], [17], [51, 2], 
        [4, 35], [5], [50, 7], [27], 
        [3, 60], [28], 
        [47], [49, 9, 61, 57, 20, 55], [23, 34], 
        [6], [10, 21, 56], [13], [16], [33], 
        [22], [44], [36], [52]]


In [86]:
print(len(leg), len(selc))

In [87]:
clusterdict = {f'c{xx}':yy for x,yy in zip(selc, leg) for xx in x}

adata.obs['L1'] = adata.obs['leiden_cons'].map(clusterdict).astype(str)
adata.obs['L1'].value_counts()


In [88]:
## seurat coordinate
ds = 0.2
fig, axes = plt.subplots(1, 3, figsize=(15, 4), dpi=300, constrained_layout=True)

for i,xx in enumerate([adata, adata_mc, adata_3c]):
    dump_embedding(xx, coord_base)
    tmp = xx.obs.copy()
    tmp['L1'] = adata.obs['L1'].copy()
    ax = axes[i]
    _ = categorical_scatter(data=tmp,
                            ax=ax,
                            coord_base=coord_base,
                            hue='L1',
                            text_anno='L1', 
                            s=ds,
                            labelsize=8,
                            max_points=None,
                            palette='tab20',
                            scatter_kws={'rasterized':True},
                            # legend_kws={'ncol':1}, 
                            # show_legend=True
                           )

for xx,ax in zip(['mCG+3C', 'mCG', '3C'], axes):
    ax.set_title(xx, fontsize=15)
    

In [89]:
# ## harmony coordinate

# fig, axes = plt.subplots(1, 3, figsize=(15, 4), dpi=300, constrained_layout=True)

# for i,xx in enumerate([adata, adata_mc, adata_3c]):
#     dump_embedding(xx, coord_base)
#     tmp = xx.obs.copy()
#     tmp['L1'] = adata.obs['L1'].copy()
#     ax = axes[i]
#     _ = categorical_scatter(data=tmp,
#                             ax=ax,
#                             coord_base=coord_base,
#                             hue='L1',
#                             text_anno='L1', 
#                             s=ds,
#                             labelsize=8,
#                             max_points=None,
#                             palette='tab20',
#                             scatter_kws={'rasterized':True},
#                             # legend_kws={'ncol':1}, 
#                             # show_legend=True
#                            )

# for xx,ax in zip(['mCG+3C', 'mCG', '3C'], axes):
#     ax.set_title(xx, fontsize=15)
    

In [90]:
adata.write_h5ad('5kCG100k3C_embed.h5ad')


In [91]:
!pwd

In [92]:
outdir = f'{indir}clustering/merged/'

In [93]:
mcad = anndata.read_h5ad(f'{outdir}5kCG.h5ad')
mcad.obs['celltype'] = mcad.obs['celltype'].astype(str)
mcad.obs['celltype'] = adata.obs.loc[mcad.obs.index, 'L1'].values


In [94]:
import os
for xx in mcad.obs['celltype'].unique():
    tmp = mcad[mcad.obs['celltype']==xx]
    x = xx.replace(' ','-').replace('/','&')
    os.makedirs(f'{outdir}L2/{x}/', exist_ok=True)
    tmp.write_h5ad(f'{outdir}L2/{x}/5kCG.h5ad')
    

In [95]:
indir = f'{ENTEX_ROOT}/tissue/'


In [96]:
file_list = glob(f'{indir}*/data/cell_table.tsv')
cell_list = pd.concat([pd.read_csv(file, sep='\t', index_col=0, header=None) for file in file_list], axis=0)


In [97]:
chromsize = pd.read_csv(f'{REF_ROOT}/hg38/fasta/hg38.main.chrom.sizes', sep='\t', index_col=0, header=None)
chromsize = chromsize.iloc[:22]


In [98]:
import os
for chrom in chromsize.index:
    file_list = glob(f'{indir}*/data/raw/{chrom}.npz')
    adata = []
    for file in file_list:
        adata.append(np.load(file)['arr_0'])
    adata = np.concatenate(adata, axis=0)
    adata = anndata.AnnData(X=adata, obs=mcad.obs.loc[cell_list.index])[mcad.obs.index]
    for xx in adata.obs['celltype'].unique():
        tmp = adata[adata.obs['celltype']==xx]
        x = xx.replace(' ','-').replace('/','&')
        os.makedirs(f'{outdir}L2/{x}/raw/', exist_ok=True)
        np.savez(f'{outdir}L2/{x}/raw/{chrom}.npz', tmp.X)
    print(chrom)


In [99]:
adata_merge = anndata.read_h5ad('merge_5kCG_echo_entex_immune.h5ad')
adata_merge

In [100]:
selc = adata.obs.index[adata.obs.index.isin(adata_merge.obs.index)]
adata_merge.obs['celltype'] = adata_merge.obs['celltype'].astype(str)
adata_merge.obs.loc[selc, 'celltype'] = adata.obs.loc[selc, 'leiden_cons'].values

In [101]:
ds = 0.5
coord_base = 'tsne'

fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=300, constrained_layout=True)

tmp = adata_merge.obs.loc[adata_merge.obs['batch']=='echo']
ax = axes[0]
ax.scatter(adata_merge.obs[f'{coord_base}_0'], adata_merge.obs[f'{coord_base}_1'], c='#e0e0e0', edgecolors='none', s=ds, rasterized=True)
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype',
                        s=ds*5,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        show_legend=True,
                        legend_kws={'ncol':1, 'fontsize':6},
                       )

                        
tmp = adata_merge.obs.loc[adata_merge.obs['batch']=='entex']
ax = axes[1]
ax.scatter(adata_merge.obs[f'{coord_base}_0'], adata_merge.obs[f'{coord_base}_1'], c='#e0e0e0', edgecolors='none', s=ds, rasterized=True)
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        show_legend=True)
